<a href="https://colab.research.google.com/github/anastasiakalyashova/python-ai-AnastasiaKalyashova/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных о горах

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas для набора данных о горах.

**Данные:**
- `data/mountains.csv` — информация о горах: название (mountain), метка (mountainLabel), координаты (coordinates), высота (elevation), тип горной породы (rockMaterialLabel)

**Что мы делаем:**
1. Клонируем ваш репозиторий GitHub в Colab
2. Читаем `mountains.csv` в pandas DataFrame
3. Очищаем и переименовываем столбцы при необходимости
4. Смотрим структуру данных, проверяем типы и делаем быструю валидацию


## 🐱 [1] Клонируем репозиторий курса в Colab

In [1]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

repo = "python-ai-AnastasiaKalyashova"  # ← изменено: имя вашего репозитория
repo_path = f"/content/{repo}"

if not os.path.exists(repo_path):
    !git clone -q https://github.com/anastasiakalyashova/python-ai-AnastasiaKalyashova.git  # ← изменено: URL вашего репозитория

if os.getcwd() != repo_path:
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

/content/python-ai-AnastasiaKalyashova
✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-AnastasiaKalyashova


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [2]:
# 🐱 Шаг 2A. Чтение mountains.csv с авто-поиском пути

import pandas as pd
import os

# Автоматически ищем файл
file_path = None
for root, dirs, files in os.walk("."):
    if "mountains.csv" in files:
        file_path = os.path.join(root, "mountains.csv")
        break

if not file_path:
    raise FileNotFoundError("❌ mountains.csv не найден! Проверьте, что файл добавлен в репозиторий или загрузите его вручную.")

# Читаем файл
df_mountains = pd.read_csv(file_path)

print(f"✅ Файл загружен из: {file_path}")
print(f"✅ Строк: {len(df_mountains)}")
print(f"✅ Столбцы: {list(df_mountains.columns)}")
print("\n📋 Первые 3 строки:")
display(df_mountains.head(3))

✅ Файл загружен из: ./data/mountains.csv
✅ Строк: 4431
✅ Столбцы: ['mountain', 'mountainLabel', 'coordinates', 'elevationMeters', 'rockMaterialLabel']

📋 Первые 3 строки:


,mountain,mountainLabel,coordinates,elevationMeters,rockMaterialLabel
0,http://www.wikidata.org/entity/Q1374,Маттерхорн,Point(7.658611111 45.976388888),4477.54,метаморфическая горная порода
1,http://www.wikidata.org/entity/Q15299,Napf,Point(7.94 47.004166666),1408.00,Конгломерат
2,http://www.wikidata.org/entity/Q513,Джомолунгма,Point(86.925 27.988055555),8848.00,горная порода


## 🧹 [2B] Очистка и переименование столбцов

В исходном CSV-файле из Викиданных есть **технические столбцы**, которые полезны для идентификации объектов, но мешают простому анализу:

- Столбец `mountain` содержит URL-ссылку на объект в Wikidata (например, `http://www.wikidata.org/entity/Q513`) — **сохраняем его для отладки**, но переименовываем в `URL`.
- Столбцы `mountainLabel` и `rockMaterialLabel` содержат читаемые названия (горы и типа породы).
- Столбец `elevationMeters` содержит высоту в метрах.

В этом шаге мы:
- переименуем столбец с URL Wikidata (`mountain` → `URL`);
- переименуем `mountainLabel → mountain`, `rockMaterialLabel → rockMaterial`;
- приведём числовой столбец `elevationMeters` (высота) к типу `float`.

При приведении к числам мы используем:

- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `.fillna(...)` — заменяет пропущенные значения при необходимости.

> ⚠️ **Важно:** столбец `URL` пригодится, если нужно будет быстро перейти к оригинальной записи горы в Викиданных. Столбец `coordinates` остаётся без изменений — его парсинг (извлечение широты/долготы из формата `Point(x y)`) можно выполнить позже при визуализации.


In [6]:
# 🧹 Шаг 2B. Очистка и переименование столбцов

# Проверяем исходные названия столбцов
original_columns = set(df_mountains.columns)

# Сценарий 1: исходные столбцы из Викиданных (ещё не переименованы)
if "mountainLabel" in original_columns and "rockMaterialLabel" in original_columns:
    df_mountains = df_mountains.rename(columns={
        "mountain": "URL",               # URL Wikidata → URL
        "mountainLabel": "mountain",     # Человекочитаемое название → mountain
        "rockMaterialLabel": "rockMaterial",  # Тип породы → rockMaterial
        "elevation": "elevationMeters",  # На случай, если вдруг был 'elevation'
    })
    print("✅ Столбцы переименованы из исходного формата Викиданных")

# Сценарий 2: столбцы уже переименованы (например, после повторного запуска)
elif "URL" in original_columns and "rockMaterial" in original_columns:
    print("⏭️ Столбцы уже переименованы, пропускаем переименование")

# Сценарий 3: неожиданный формат — выводим диагностику
else:
    print("⚠️ Неожиданный формат столбцов. Текущие столбцы:")
    print(list(df_mountains.columns))
    print("\nПопытка автоматической адаптации...")
    # Пытаемся найти столбец с высотой по ключевому слову
    elevation_col = next((col for col in df_mountains.columns if 'elev' in col.lower()), None)
    if elevation_col and elevation_col != 'elevationMeters':
        df_mountains = df_mountains.rename(columns={elevation_col: 'elevationMeters'})
        print(f"   → Столбец '{elevation_col}' переименован в 'elevationMeters'")

print("\n✅ Текущие столбцы:", list(df_mountains.columns))

# Приводим высоту к числовому типу (с обработкой ошибок)
if 'elevationMeters' in df_mountains.columns:
    df_mountains['elevationMeters'] = pd.to_numeric(df_mountains['elevationMeters'], errors='coerce')
    print("Тип данных в столбце elevationMeters:", df_mountains["elevationMeters"].dtype)
else:
    print("⚠️ Столбец с высотой не найден!")

# Нормализуем столбец с породой (если существует)
if 'rockMaterial' in df_mountains.columns:
    print("\nДо нормализации:", df_mountains['rockMaterial'].nunique(), "уникальных значений")
    df_mountains['rockMaterial'] = (
        df_mountains['rockMaterial']
        .str.lower()
        .str.strip()
        .fillna("неизвестно")  # обрабатываем пропуски
    )
    print("После нормализации:", df_mountains['rockMaterial'].nunique(), "уникальных значений")
    print("\nТоп-10 пород после нормализации:")
    print(df_mountains['rockMaterial'].value_counts().head(10))
else:
    print("\n⚠️ Столбец 'rockMaterial' отсутствует. Доступные столбцы:", list(df_mountains.columns))

⏭️ Столбцы уже переименованы, пропускаем переименование

✅ Текущие столбцы: ['URL', 'mountain', 'coordinates', 'elevationMeters', 'rockMaterial']
Тип данных в столбце elevationMeters: float64

До нормализации: 108 уникальных значений
После нормализации: 108 уникальных значений

Топ-10 пород после нормализации:
rockMaterial
известняк                  838
песчаник                   520
гранит                     357
мергель                    300
конгломерат                294
lutite                     230
доломит                    183
андезит                    174
осадочная горная порода    162
базальт                    155
Name: count, dtype: int64


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame с данными о горах:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по высоте (`elevationMeters`) — минимальная, максимальная, средняя высота и т.д.

Для удобства используем функцию `show_info(df, name)`, чтобы компактно вывести информацию о таблице.

In [8]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

# 🔍 Шаг 3. Обзор данных

show_info(df_mountains, "Горы (df_mountains)")

# 📈 Базовая статистика по высоте
elevation_col = next((col for col in df_mountains.columns if 'elev' in col.lower()), None)

if elevation_col:
    print(f"\n📈 Статистика по высоте ({elevation_col}):")
    print(df_mountains[elevation_col].describe())
else:
    print("\n⚠️ Столбец с высотой не найден. Доступные столбцы:")
    print(df_mountains.columns.tolist())


📊 Горы (df_mountains)
Размер: (4431, 5)
Столбцы: URL, mountain, coordinates, elevationMeters, rockMaterial

Первые строки:
                                     URL     mountain  \
0   http://www.wikidata.org/entity/Q1374   Маттерхорн   
1  http://www.wikidata.org/entity/Q15299         Napf   
2    http://www.wikidata.org/entity/Q513  Джомолунгма   
3    http://www.wikidata.org/entity/Q513  Джомолунгма   
4   http://www.wikidata.org/entity/Q7093       Неулос   

                       coordinates  elevationMeters  \
0  Point(7.658611111 45.976388888)          4477.54   
1         Point(7.94 47.004166666)          1408.00   
2       Point(86.925 27.988055555)          8848.00   
3       Point(86.925 27.988055555)          8848.00   
4  Point(2.947110073 42.482101374)          1257.00   

                    rockMaterial  
0  метаморфическая горная порода  
1                    конгломерат  
2                  горная порода  
3                            лёд  
4         кристаллические с

## ❄️ [4] Уникальный анализ: «многослойность» гор и «ледяной пояс» Земли

Ваши данные обладают редкой особенностью — **геологическая «многослойность»**: одна и та же гора может иметь несколько записей с разными типами пород и материалов. Например, Джомолунгма представлена двумя слоями: «горная порода» у основания и «лёд» на вершине.

В этом шаге мы исследуем два уникальных явления:

### 🗻 1. Геологическая «многослойность»
- Сколько **уникальных гор** скрыто в записях?
- Какие горы имеют **наибольшее разнообразие материалов** (геологическая сложность)?
- Есть ли горы, представленные 3+ разными породами — признак сложной структуры?

### 🧊 2. «Ледяной пояс» Земли
Лёд не встречается равномерно — он концентрируется в определённом высотном диапазоне. Мы выявим:
- На какой **минимальной высоте** начинает формироваться постоянный ледник?
- В каком **диапазоне высот** лёд встречается чаще всего («ледяной пояс»)?
- Есть ли горы с льдом **ниже 4 000 м** — признак полярного климата?

> 💡 **Интересный факт**: В Гималаях ледники начинаются ~5 000 м, а в Антарктиде — уже на уровне моря. Анализ высоты появления льда поможет косвенно определить географическое положение гор!



In [10]:
# ❄️ Шаг 4. Анализ «многослойности» и «ледяного пояса»

print("=" * 70)
print("🏔️  ЧАСТЬ 1: Геологическая «многослойность» гор")
print("=" * 70)

# 🔧 Парсим координаты из формата WKT Point(x y)
if 'coordinates' in df_mountains.columns:
    coords = df_mountains['coordinates'].str.extract(r'Point\(([^ ]+)\s+([^)]+)\)')
    df_mountains['lon'] = pd.to_numeric(coords[0], errors='coerce')
    df_mountains['lat'] = pd.to_numeric(coords[1], errors='coerce')
    print("✅ Координаты распарсены:")
    print(df_mountains[['mountain', 'lon', 'lat']].head(3))
else:
    print("⚠️ Столбец 'coordinates' не найден")

# 1.1 Сравнение уникальных гор vs общего числа записей
total_records = len(df_mountains)
unique_mountains = df_mountains["URL"].nunique() if "URL" in df_mountains.columns else df_mountains["mountain"].nunique()
print(f"\n📊 Всего записей: {total_records}")
print(f"📊 Уникальных гор: {unique_mountains}")
print(f"📊 Среднее число записей на гору: {total_records / unique_mountains:.2f}")
print(f"   → В среднем у каждой горы есть данные о {total_records / unique_mountains:.1f} типах материалов!")

# 1.2 Группировка по уникальному идентификатору горы (URL)
group_col = "URL" if "URL" in df_mountains.columns else "mountain"

df_unique = (
    df_mountains
    .groupby(group_col)
    .agg(
        mountain=('mountain', 'first'),          # Название горы
        lon=('lon', 'first'),                    # Долгота
        lat=('lat', 'first'),                    # Широта
        elevation=('elevationMeters', 'first'),  # Высота
        rock_count=('rockMaterial', 'nunique'),  # Число уникальных пород
        rocks=('rockMaterial', lambda x: list(x.unique())),  # Список уникальных пород
    )
    .reset_index()
)
print(f"\n✅ df_unique: {len(df_unique)} уникальных гор")
print(f"   (было {len(df_mountains)} строк в длинном формате)")
print(f"\nСреднее число пород на гору: {df_unique['rock_count'].mean():.2f}")

# Статистика по высоте
if 'elevation' in df_unique.columns:
    print("\n📈 Статистика по высоте (по уникальным горам):")
    print(df_unique['elevation'].describe())

# 1.3 Распределение «многослойности»
layer_counts = df_mountains.groupby(group_col).size()
distribution = layer_counts.value_counts().sort_index()
print("\n📈 Распределение записей на гору:")
for layers, count in distribution.head(6).items():
    print(f"   {layers} слоя(ев): {count} гор(ы)")

# 1.4 Топ-10 гор с наибольшим разнообразием материалов
top_complex = (df_mountains.groupby(group_col)
               .agg(
                   mountain=('mountain', 'first'),
                   elevation=('elevationMeters', 'first'),
                   rock_count=('rockMaterial', 'nunique')
               )
               .sort_values("rock_count", ascending=False)
               .head(10))

print("\n🏆 Топ-10 гор с наибольшим разнообразием материалов:")
print(top_complex[['mountain', 'elevation', 'rock_count']].to_string(index=False))

# 1.5 Проверка на аномалии в высоте
MAX_REAL = 8849  # Высота Джомолунгмы в метрах
if 'elevationMeters' in df_mountains.columns:
    anomalies_high = df_mountains[df_mountains['elevationMeters'] > MAX_REAL]
    print(f"\n⚠️  Гор с высотой > {MAX_REAL} м: {len(anomalies_high)}")
    if len(anomalies_high) > 0:
        print(anomalies_high[['mountain', 'elevationMeters', 'rockMaterial']].to_string(index=False))

    anomalies_neg = df_mountains[df_mountains['elevationMeters'] < 0]
    print(f"\n⚠️  Гор с отрицательной высотой: {len(anomalies_neg)}")
    if len(anomalies_neg) > 0:
        print(anomalies_neg[['mountain', 'elevationMeters']].to_string(index=False))

print("\n" + "=" * 70)
print("🧊 ЧАСТЬ 2: «Ледяной пояс» Земли — где живёт лёд?")
print("=" * 70)

# 2.1 Поиск всех вариантов написания «лёд»
ice_names = {"лёд", "лед", "льда", "ледник", "ice", "snow", "glacier"}
df_mountains["is_ice"] = df_mountains["rockMaterial"].str.lower().isin(ice_names)

ice_records = df_mountains[df_mountains["is_ice"]]
total_ice = len(ice_records)
print(f"\n❄️  Записей с льдом: {total_ice} из {total_records} ({total_ice/total_records*100:.1f}%)")

if total_ice > 0 and 'elevationMeters' in df_mountains.columns:
    # 2.2 Минимальная и максимальная высота с льдом
    min_ice = ice_records["elevationMeters"].min()
    max_ice = ice_records["elevationMeters"].max()
    print(f"   Минимальная высота с льдом: {min_ice:.0f} м")
    print(f"   Максимальная высота с льдом: {max_ice:.0f} м")

    # 2.3 Высотные диапазоны для анализа «ледяного пояса»
    bins = [-100, 0, 2000, 4000, 5000, 6000, 7000, 8000, 10000]
    labels = ["< 0", "0–2 км", "2–4 км", "4–5 км", "5–6 км", "6–7 км", "7–8 км", "8+ км"]

    df_mountains["height_bin"] = pd.cut(df_mountains["elevationMeters"], bins=bins, labels=labels, right=False)
    ice_by_bin = (df_mountains.groupby("height_bin")
                  .agg(total=("is_ice", "size"), ice=("is_ice", "sum"))
                  .assign(ice_pct=lambda x: (x["ice"] / x["total"] * 100).round(1))
                  .sort_index())

    print("\n📊 Распределение льда по высотным диапазонам:")
    print(ice_by_bin[["total", "ice", "ice_pct"]].rename(columns={
        "total": "Всего записей",
        "ice": "С льдом",
        "ice_pct": "% с льдом"
    }).to_string())

    # 2.4 Определение «ледяного пояса»
    peak_bin = ice_by_bin["ice_pct"].idxmax()
    peak_value = ice_by_bin.loc[peak_bin, "ice_pct"]
    print(f"\n🎯 «Ледяной пояс» Земли: {peak_bin} — лёд встречается в {peak_value}% записей!")

    # 2.5 Горы с льдом ниже 4000 м (полярные регионы)
    low_ice = ice_records[ice_records["elevationMeters"] < 4000]
    if len(low_ice) > 0:
        print(f"\n🔍 Необычные находки: {len(low_ice)} записей с льдом ниже 4000 м")
        print("   Примеры:")
        for _, row in low_ice.head(5).iterrows():
            print(f"   • {row['mountain']} ({row['elevationMeters']:.0f} м) — {row['rockMaterial']}")
else:
    print("   ⚠️  Лёд не обнаружен в данных или отсутствует столбец высоты.")

# Очистка временных столбцов
df_mountains.drop(columns=["is_ice", "height_bin"], inplace=True, errors="ignore")

🏔️  ЧАСТЬ 1: Геологическая «многослойность» гор
✅ Координаты распарсены:
      mountain        lon        lat
0   Маттерхорн   7.658611  45.976389
1         Napf   7.940000  47.004167
2  Джомолунгма  86.925000  27.988056

📊 Всего записей: 4431
📊 Уникальных гор: 2915
📊 Среднее число записей на гору: 1.52
   → В среднем у каждой горы есть данные о 1.5 типах материалов!

✅ df_unique: 2915 уникальных гор
   (было 4431 строк в длинном формате)

Среднее число пород на гору: 1.43

📈 Статистика по высоте (по уникальным горам):
count    2915.000000
mean     1641.963647
std      1130.311055
min       -39.000000
25%       733.000000
50%      1397.000000
75%      2456.500000
max      8848.000000
Name: elevation, dtype: float64

📈 Распределение записей на гору:
   1 слоя(ев): 1840 гор(ы)
   2 слоя(ев): 698 гор(ы)
   3 слоя(ев): 341 гор(ы)
   4 слоя(ев): 28 гор(ы)
   5 слоя(ев): 2 гор(ы)
   6 слоя(ев): 3 гор(ы)

🏆 Топ-10 гор с наибольшим разнообразием материалов:
         mountain  elevation  rock_c

/tmp/ipykernel_3247/2928503864.py:107: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ice_by_bin = (df_mountains.groupby("height_bin")


## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных** гор и типов пород есть в данных;
- **какие горы самые высокие** (Топ‑5 по высоте);
- **какие типы пород встречаются чаще всего** (Топ‑10 по числу записей);
- **диапазон высот** гор в датасете.

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому
`df_mountains["rockMaterial"].value_counts().head(10)` даёт **Топ‑10 типов пород по частоте**.

Метод `.nlargest(n)` возвращает топ-N значений по указанному столбцу, поэтому
`df_mountains.nlargest(5, "elevationMeters")` даёт **5 самых высоких гор**.


In [11]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных")

# Датасет: горы
print("\n📊 Датасет: Горы (df_mountains)")
print("Уникальных гор:", df_mountains["mountain"].nunique())
print("Уникальных типов пород/материалов:", df_mountains["rockMaterial"].nunique())

# Проверяем наличие столбца с высотой
elevation_col = next((col for col in df_mountains.columns if 'elev' in col.lower()), None)

if elevation_col:
    print(f"\n📈 Диапазон высот ({elevation_col}):")
    print(f"Минимальная: {df_mountains[elevation_col].min()} м")
    print(f"Максимальная: {df_mountains[elevation_col].max()} м")
    print(f"Средняя: {df_mountains[elevation_col].mean():.2f} м")

    print("\n🏆 Топ-5 самых высоких гор:")
    top_mountains = df_mountains.nlargest(5, elevation_col)[["mountain", elevation_col, "rockMaterial"]]
    print(top_mountains.to_string(index=False))

    print("\n📊 Топ-10 типов пород/материалов по частоте:")
    print(df_mountains["rockMaterial"].value_counts().head(10))

    print(f"\n📈 Распределение высот (квантили):")
    print(df_mountains[elevation_col].describe())
else:
    print("\n⚠️ Столбец с высотой не найден!")
    print("Доступные столбцы:", list(df_mountains.columns))


🔍 Быстрая проверка данных

📊 Датасет: Горы (df_mountains)
Уникальных гор: 2831
Уникальных типов пород/материалов: 108

📈 Диапазон высот (elevationMeters):
Минимальная: -39.0 м
Максимальная: 8850.0 м
Средняя: 1547.89 м

🏆 Топ-5 самых высоких гор:
   mountain  elevationMeters  rockMaterial
Джомолунгма         8850.000 горная порода
Джомолунгма         8850.000           лёд
Джомолунгма         8848.860 горная порода
Джомолунгма         8848.860           лёд
Джомолунгма         8848.344 горная порода

📊 Топ-10 типов пород/материалов по частоте:
rockMaterial
известняк                  838
песчаник                   520
гранит                     357
мергель                    300
конгломерат                294
lutite                     230
доломит                    183
андезит                    174
осадочная горная порода    162
базальт                    155
Name: count, dtype: int64

📈 Распределение высот (квантили):
count    4431.000000
mean     1547.893639
std      1134.510703
min 

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали CSV-файл `mountains.csv` из папки `data/`
- ✅ Переименовали столбцы для удобства (`mountainLabel → mountain`, `rockMaterialLabel → rockMaterial`, `elevationMeters` приведён к числовому типу)
- ✅ Распарсили координаты из формата WKT (`Point(x y)`) в отдельные столбцы `lon` и `lat`
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных гор и типов пород
  - диапазон высот (минимум, максимум, среднее)
  - топ-5 самых высоких гор
  - топ-10 типов пород по частоте
  - проверка на аномалии в высоте (отрицательные значения, выше Джомолунгмы)

Теперь у нас есть **аккуратная, проверенная таблица** с данными о горах, с которой удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **эти же данные** для:
- более глубокого анализа (геологическая «многослойность», «ледяной пояс» Земли),
- и построения визуализаций (карты, гистограммы высот, распределение пород). 🎨